# 04 Notebook: Model Testing (Live)

Loading the saved ML model to detect blinks in real-time from the webcam.

In [ ]:
import cv2
import time
import numpy as np
import pandas as pd
import joblib
import mediapipe as mp
import matplotlib.pyplot as plt
from IPython.display import display, Image, clear_output

# Load Models
try:
    model = joblib.load('blink_model.pkl')
    scaler = joblib.load('scaler.pkl')
    print("Loaded model and scaler successfully.")
except Exception as e:
    print("Error loading models:", e)

# Setup MediaPipe
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=False, max_num_faces=1, refine_landmarks=True)

# Function to extract features matching exactly Notebook 2
def extract_single_frame_features(image, results):
    if not results.multi_face_landmarks: return None
    landmarks = results.multi_face_landmarks[0].landmark
    h, w, _ = image.shape
    pts = [(int(pt.x * w), int(pt.y * h)) for pt in landmarks]
    
    def euclidean_distance(p1, p2):
        return np.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)
    
    LEFT_EYE = [33, 160, 158, 133, 153, 144]
    RIGHT_EYE = [362, 385, 387, 263, 373, 380]
    
    l_p0, l_p1, l_p2, l_p3, l_p4, l_p5 = [pts[i] for i in LEFT_EYE]
    r_p0, r_p1, r_p2, r_p3, r_p4, r_p5 = [pts[i] for i in RIGHT_EYE]
    
    lear = (euclidean_distance(l_p1, l_p5) + euclidean_distance(l_p2, l_p4)) / (2.0 * euclidean_distance(l_p0, l_p3))
    rear = (euclidean_distance(r_p1, r_p5) + euclidean_distance(r_p2, r_p4)) / (2.0 * euclidean_distance(r_p0, r_p3))
    avg_ear = (lear + rear) / 2.0
    
    l_w = euclidean_distance(l_p0, l_p3)
    l_h = (euclidean_distance(l_p1, l_p5) + euclidean_distance(l_p2, l_p4)) / 2.0
    r_w = euclidean_distance(r_p0, r_p3)
    r_h = (euclidean_distance(r_p1, r_p5) + euclidean_distance(r_p2, r_p4)) / 2.0
    
    l_op = l_h / (l_w + 1e-6)
    r_op = r_h / (r_w + 1e-6)
    
    pv = (l_h + r_h) / 2.0 
    
    l_eb = pts[105]
    r_eb = pts[334]
    l_ed = euclidean_distance(l_p1, l_eb)
    r_ed = euclidean_distance(r_p1, r_eb)
    
    features = [lear, rear, avg_ear, l_w, l_h, r_w, r_h, l_op, r_op, l_h, r_h, pv, l_ed, r_ed]
    return np.array(features).reshape(1, -1), avg_ear


In [ ]:
# Live Testing
cap = cv2.VideoCapture(0)

history_ear = []
history_conf = []
history_preds = []
timestamps = []

last_state = 0
blink_timestamps = []
total_frames = 0
single_blinks = 0
double_blinks = 0
start_time = time.time()
test_duration = 60 # Run for 60 seconds

print("Starting live test for 60 seconds...")
try:
    while time.time() - start_time < test_duration:
        ret, frame = cap.read()
        if not ret: break
        
        loop_start = time.time()
        
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(rgb_frame)
        
        pred_label = "Waiting..."
        conf = 0.0
        ear = 0.0
        
        if results.multi_face_landmarks:
            feat_data = extract_single_frame_features(frame, results)
            if feat_data is not None:
                features, ear = feat_data
                
                scaled_feat = scaler.transform(features)
                pred_class = model.predict(scaled_feat)[0]
                if hasattr(model, 'predict_proba'):
                    conf = np.max(model.predict_proba(scaled_feat)) * 100
                else: 
                    conf = 100.0
                
                classes = {0: "OPEN", 1: "CLOSED"}
                pred_label = classes.get(pred_class, "UNKNOWN")
                
                current_time = time.time()
                if last_state == 1 and pred_class == 0:
                    if len(blink_timestamps) > 0 and (current_time - blink_timestamps[-1]) < 0.6:
                        double_blinks += 1
                        single_blinks -= 1
                        blink_timestamps = []
                    else:
                        single_blinks += 1
                        blink_timestamps.append(current_time)
                last_state = pred_class
                
                history_ear.append(ear)
                history_conf.append(conf)
                history_preds.append(pred_label)
                timestamps.append(time.time() - start_time)
        
        fps = 1.0 / (time.time() - loop_start + 1e-6)
        total_frames += 1
        
        # Overlays
        cv2.putText(frame, f"Prediction: {pred_label}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.putText(frame, f"Conf: {conf:.1f}%", (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
        cv2.putText(frame, f"EAR: {ear:.3f}", (10, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 100, 100), 2)
        cv2.putText(frame, f"FPS: {fps:.1f}", (10, 130), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        cv2.putText(frame, f"Blinks: {single_blinks+double_blinks}", (10, 160), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 255), 2)
        
        cv2.imshow("Live Testing (60s) - Press Q to stop", frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

finally:
    cap.release()
    cv2.destroyAllWindows()
    print("Test Finished.")

In [ ]:
try:
    single_blinks
    double_blinks
    history_conf
    history_preds
    timestamps
    total_frames
    test_duration
    history_ear
except NameError:
    single_blinks = 0
    double_blinks = 0
    history_conf = []
    history_preds = []
    timestamps = []
    total_frames = 1
    test_duration = 1
    history_ear = []

# Final Report
tot_blinks = single_blinks + double_blinks
import numpy as np
avg_conf = np.mean(history_conf) if history_conf else 0
avg_fps = total_frames / test_duration

print('='*30)
print('FINAL REPORT')
print('='*30)
print(f'Total Frames Processed: {total_frames}')
print(f'Total Blinks Detected: {tot_blinks} frames')
print(f'Single Blink Count: {single_blinks} frames')
print(f'Double Blink Count: {double_blinks} frames')
print(f'Average Confidence: {avg_conf:.1f}%')
print(f'Average FPS: {avg_fps:.1f}')

import matplotlib.pyplot as plt
plt.figure(figsize=(15, 8))

# 1. EAR over time
if timestamps:
    plt.subplot(2, 2, 1)
    plt.plot(timestamps, history_ear, color='purple')
    plt.title('EAR Value Over Time')
    plt.xlabel('Seconds')
    plt.ylabel('EAR')

# 2. Confidence over time
if timestamps:
    plt.subplot(2, 2, 2)
    plt.plot(timestamps, history_conf, color='orange')
    plt.title('Prediction Confidence Over Time')
    plt.xlabel('Seconds')
    plt.ylabel('Confidence %')

# 3. Pie Chart
plt.subplot(2, 2, 3)
closed_frames = history_preds.count('CLOSED')
open_frames = history_preds.count('OPEN')
labels = ['Open Frames', 'Closed Frames']
sizes = [open_frames, closed_frames]
colors = ['#ff9999','#66b3ff']
if sum(sizes) > 0:
    plt.pie([s for s in sizes if s > 0], labels=[l for l,s in zip(labels, sizes) if s>0], autopct='%1.1f%%', colors=colors)
plt.title('Class Distribution (Frames)')

# 4. Timeline Strip Plot
if timestamps:
    plt.subplot(2, 2, 4)
    history_y = [1 if p == 'CLOSED' else 0 for p in history_preds]
    plt.scatter(timestamps, history_y, alpha=0.5, s=10)
    plt.yticks([0, 1], ['Open', 'Closed'])
    plt.title('Predictions Timeline')
    plt.xlabel('Seconds')

plt.tight_layout()
plt.show()
